In [ ]:
# !pip --version

pip 26.2.1 from D:\kbh\ex0916\.venv\Lib\site-packages\pip (python 3.12)



In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [5]:
llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")
llm.invoke("이탈리아의 수도는 뭐야")

AIMessage(content='이탈리아의 수도는 로마(Rome)입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 15, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_27270670a9', 'id': 'chatcmpl-EOYZ6VicX7M2HEjGBss6kZVs1NG3f', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7ca-4b05-7341-913e-087b027a0506-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 13, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [7]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [8]:
llm_with_structered = ChatOpenAI(
    temperature=0, model="gpt-4.1-mini"
).with_structured_output(EmailSummary)

In [9]:
answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='김철수 상무는 이은채 대리에게 바이크코퍼레이션이 자전거 제조 및 유통 분야에서의 경험을 바탕으로 귀사의 신규 자전거 "ZENESIS"에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하며, 협력 가능성 논의를 위해 1월 15일 화요일 오전 10시에 미팅을 제안합니다.', date='2024-01-08')

In [10]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

In [13]:
output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

In [14]:
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [15]:
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

In [16]:
prompt2 = PromptTemplate(
    template="다섯가지 {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions}, 
)

In [17]:
model = ChatOpenAI(temperature=0)

chain = prompt | model | output_parser

In [18]:
chain.invoke({"subject": "호주 관광명소"})

['시드니 오페라하우스', '그레이트 오션 로드', '울룰루', '그레이트 베리어 리프', '블루 마운틴즈']

In [20]:
model = ChatOpenAI(temperature=0)

chain2 = prompt2 | model | output_parser

chain2.invoke({"subject": "인도 관광명소"})

['타지마할', '자이푸르', '고아', '코찌코데', '바라나시']

In [21]:
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)

['경복궁']
['남산타워']
['부산 해운대해수욕장']
['제주도 성산일출봉']
['경주 불국사temples']
